# Demo completa Sudoku tramite archivio persistente

Questo notebook usa `sudoku_archive.py` come unica interfaccia principale per i dati.
Le analisi `deep`, `profile` e `superficial` vengono salvate in varianti distinte,
quindi un risultato già calcolato può essere riutilizzato nelle esecuzioni successive.

Il flusso verifica:

1. elenco, selezione, salvataggio e caricamento dei Sudoku;
2. analisi persistenti nelle tre modalità;
3. cache in memoria e caricamento da disco;
4. equivalenza della catena risolutiva fra le modalità;
5. dataframe e visualizzazioni aggiornate;
6. quattro heatmap principali;
7. confronto fra conclusioni, risultati distinti e prove;
8. presenza delle varianti nell archivio.

La modalità canonica e predefinita resta `deep`.


## 0. Configurazione

Per riutilizzare tutto ciò che è già stato analizzato, lascia
`FORCE_RECALCULATION = False`.


In [ ]:
CREATE_NEW_PUZZLE = False
TARGET_CLUES = 30

SELECTION_METHOD = "latest"
SELECTION_NUMBER = 20

FORCE_RECALCULATION = False
PROFILE_DIFFICULTY_WINDOW = 1.0

RUN_ALL_METRIC_HEATMAPS = True
RUN_ALL_SCALE_HEATMAPS = True
RUN_TECHNIQUE_DETAIL_HEATMAP = True

RUN_ARCHIVE_BATCH = False
ARCHIVE_BATCH_LIMIT = 10


## 1. Import e ricaricamento dei moduli


In [ ]:
import importlib
import inspect
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

import sudoku_techniques as st
import sudoku_solver as ss
import sudoku_archive as sa
import sudoku_visualization as sv
import sudoku_generator as sg

importlib.invalidate_caches()
st = importlib.reload(st)
ss = importlib.reload(ss)
sa = importlib.reload(sa)
sv = importlib.reload(sv)
sg = importlib.reload(sg)

plt.rcParams["figure.figsize"] = (8, 6)
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 180)

print("Moduli caricati.")
print("ANALYSIS_VERSION:", sa.ANALYSIS_VERSION)
print("Modalità predefinita archivio:", sa.DEFAULT_ANALYSIS_MODE)


## 2. Controllo delle nuove interfacce

Il notebook richiede la versione dell archivio che distingue le varianti di analisi.


In [ ]:
archive_signature = inspect.signature(sa.analyse_puzzle_cached)
load_signature = inspect.signature(sa.load_analysis)

required_cached_parameters = {
    "analysis_mode",
    "profile_difficulty_window",
    "force",
}
required_load_parameters = {
    "analysis_mode",
    "profile_difficulty_window",
}

missing_cached = required_cached_parameters - set(archive_signature.parameters)
missing_load = required_load_parameters - set(load_signature.parameters)

assert not missing_cached, (
    "sudoku_archive.py non aggiornato. Parametri mancanti in "
    f"analyse_puzzle_cached: {sorted(missing_cached)}"
)
assert not missing_load, (
    "sudoku_archive.py non aggiornato. Parametri mancanti in "
    f"load_analysis: {sorted(missing_load)}"
)
assert sa.ANALYSIS_VERSION >= 8, (
    "Incrementa ANALYSIS_VERSION per invalidare le vecchie analisi."
)

print("Interfacce archivio aggiornate: OK")
print("analyse_puzzle_cached", archive_signature)
print("load_analysis", load_signature)


## 3. Utility del notebook


In [ ]:
def timed_call(function, *args, **kwargs):
    start = time.perf_counter()
    result = function(*args, **kwargs)
    return result, time.perf_counter() - start


def chain_signature(analysis):
    return [
        (
            move["technique"],
            float(move["difficulty"]),
            tuple(sorted(tuple(item) for item in move.get("placements", []))),
            tuple(sorted(tuple(item) for item in move.get("eliminations", []))),
        )
        for move in analysis.get("chain", [])
    ]


def validate_analysis(analysis, expected_mode, expected_window=None):
    assert analysis["analysis_mode"] == expected_mode

    if expected_mode == "profile":
        assert abs(
            float(analysis["profile_difficulty_window"])
            - float(expected_window)
        ) <= 1e-12
    else:
        assert analysis.get("profile_difficulty_window") is None

    for step_index, step in enumerate(analysis.get("chain", []), start=1):
        assert "availability" in step, (
            f"Step {step_index}: availability mancante."
        )
        assert "frontier" in step["availability"]
        assert "n_conclusions" in step
        assert "n_best_conclusions" in step
        assert "n_proofs" in step

    return True


def analysis_row(analysis, elapsed=None):
    chain = analysis.get("chain", [])
    grading = analysis.get("grading", {})
    scanned = [
        step.get("analysis_scope", {}).get("scanned_function_count")
        for step in chain
    ]
    scanned = [value for value in scanned if value is not None]

    return {
        "modalita": analysis.get("analysis_mode"),
        "finestra_profile": analysis.get("profile_difficulty_window"),
        "tempo_s": round(elapsed, 5) if elapsed is not None else None,
        "stato": analysis.get("status"),
        "step": len(chain),
        "difficolta_massima": grading.get("max_difficulty"),
        "difficolta_percepita": grading.get("perceived_difficulty"),
        "funzioni_medie_scandite": (
            round(float(np.mean(scanned)), 2)
            if scanned
            else None
        ),
        "conclusioni_osservate": sum(
            int(step.get("n_conclusions", 0))
            for step in chain
        ),
        "conclusioni_minime": sum(
            int(step.get("n_best_conclusions", 0))
            for step in chain
        ),
        "risultati_distinti": sum(
            int(step.get("n_distinct_outcomes", 0))
            for step in chain
        ),
        "prove": sum(
            int(step.get("n_proofs", 0))
            for step in chain
        ),
    }


def generate_demo_puzzle(target_clues, rng):
    candidates = [
        getattr(sg, "generate_unique_puzzle", None),
        getattr(sg, "generate_sudoku", None),
    ]
    candidates = [function for function in candidates if callable(function)]

    if not candidates:
        raise AttributeError(
            "Non trovo generate_unique_puzzle o generate_sudoku."
        )

    function = candidates[0]
    parameters = inspect.signature(function).parameters
    kwargs = {}

    for name in ("target_clues", "clues", "n_clues"):
        if name in parameters:
            kwargs[name] = target_clues
            break

    if "rng" in parameters:
        kwargs["rng"] = rng
    elif "seed" in parameters:
        kwargs["seed"] = rng.randrange(1, 10**9)

    result = function(**kwargs)

    if isinstance(result, dict):
        puzzle = next(
            result[key]
            for key in ("puzzle", "grid", "sudoku")
            if result.get(key) is not None
        )
    elif isinstance(result, tuple):
        puzzle = result[0]
    else:
        puzzle = result

    return np.asarray(puzzle, dtype=int).reshape(9, 9), function.__name__


## 4. Stato corrente dell archivio

`list_sudokus` espone anche le varianti già disponibili per ogni Sudoku.


In [ ]:
archive_rows = sa.list_sudokus(
    number=SELECTION_NUMBER,
    method=SELECTION_METHOD,
)
archive_dataframe = pd.DataFrame(archive_rows)

display(archive_dataframe)
print("Sudoku trovati:", len(archive_rows))


## 5. Creazione facoltativa o selezione del Sudoku

Il percorso normale usa l ultimo Sudoku già presente. La generazione è soltanto
un opzione di test e salva subito il puzzle tramite l archivio.


In [ ]:
if CREATE_NEW_PUZZLE:
    seed = random.SystemRandom().randrange(1, 10**9)
    rng = random.Random(seed)
    generated_grid, generator_name = generate_demo_puzzle(
        TARGET_CLUES,
        rng,
    )

    stored_puzzle = sa.save_sudoku(
        generated_grid,
        name=f"analysis_demo_{seed}",
        metadata={
            "seed": seed,
            "generator": generator_name,
            "target_clues": TARGET_CLUES,
            "purpose": "demo archivio analisi multiple",
        },
    )
else:
    if not archive_rows:
        raise FileNotFoundError(
            "L archivio è vuoto. Imposta CREATE_NEW_PUZZLE = True."
        )

    stored_puzzle = sa.load_sudoku(archive_rows[0]["id"])

puzzle_reference = stored_puzzle["id"]
loaded_puzzle = stored_puzzle["grid"]

print("ID:", stored_puzzle["id"])
print("Nome:", stored_puzzle["name"])
print("Indizi:", stored_puzzle["clues"])
print("Percorso:", stored_puzzle["path"])

sv.draw_grid(
    loaded_puzzle,
    title=f"{stored_puzzle['name']} - puzzle selezionato",
)
plt.show()


## 6. Analisi persistenti nelle tre modalità

Tutte le chiamate passano da `analyse_puzzle_cached`. Se il risultato corrente
esiste già su disco, non viene ricalcolato.


In [ ]:
analysis_specs = {
    "deep": {},
    "profile": {
        "profile_difficulty_window": PROFILE_DIFFICULTY_WINDOW,
    },
    "superficial": {},
}

mode_results = {}
mode_times = {}

for mode, extra_kwargs in analysis_specs.items():
    result, elapsed = timed_call(
        sa.analyse_puzzle_cached,
        puzzle_reference,
        analysis_mode=mode,
        force=FORCE_RECALCULATION,
        **extra_kwargs,
    )

    validate_analysis(
        result,
        expected_mode=mode,
        expected_window=(
            PROFILE_DIFFICULTY_WINDOW
            if mode == "profile"
            else None
        ),
    )

    mode_results[mode] = result
    mode_times[mode] = elapsed

comparison_dataframe = pd.DataFrame([
    analysis_row(mode_results[mode], mode_times[mode])
    for mode in ("deep", "profile", "superficial")
])

display(comparison_dataframe)


## 7. Invarianza della catena risolutiva

La modalità cambia l inventario raccolto, non la mossa scelta.


In [ ]:
deep_signature = chain_signature(mode_results["deep"])

for mode in ("profile", "superficial"):
    same_chain = chain_signature(mode_results[mode]) == deep_signature
    same_grid = np.array_equal(
        mode_results[mode]["solved_grid"],
        mode_results["deep"]["solved_grid"],
    )

    print(mode, "catena identica:", same_chain)
    print(mode, "griglia finale identica:", same_grid)

    assert same_chain
    assert same_grid


## 8. Verifica della cache in memoria

Ogni variante deve avere una propria voce e restituire lo stesso oggetto Python.


In [ ]:
for mode, extra_kwargs in analysis_specs.items():
    cached_result, elapsed = timed_call(
        sa.analyse_puzzle_cached,
        puzzle_reference,
        analysis_mode=mode,
        force=False,
        **extra_kwargs,
    )

    print(
        mode,
        "stesso oggetto:",
        cached_result is mode_results[mode],
        "tempo:",
        round(elapsed, 6),
        "s",
    )

    assert cached_result is mode_results[mode]


## 9. Verifica del caricamento da disco

Il modulo archivio viene ricaricato per svuotare la cache in memoria. Le tre
varianti vengono poi lette con `load_analysis`.


In [ ]:
sa = importlib.reload(sa)
disk_results = {}

for mode, extra_kwargs in analysis_specs.items():
    result, elapsed = timed_call(
        sa.load_analysis,
        puzzle_reference,
        analysis_mode=mode,
        **extra_kwargs,
    )

    validate_analysis(
        result,
        expected_mode=mode,
        expected_window=(
            PROFILE_DIFFICULTY_WINDOW
            if mode == "profile"
            else None
        ),
    )

    disk_results[mode] = result

    print(
        mode,
        "caricato da disco in",
        round(elapsed, 6),
        "s",
    )

    assert chain_signature(result) == deep_signature

mode_results = disk_results


## 10. Varianti registrate nell archivio


In [ ]:
updated_rows = sa.list_sudokus(method="latest")
updated_dataframe = pd.DataFrame(updated_rows)
current_row = updated_dataframe[
    updated_dataframe["id"] == puzzle_reference
]

display(current_row)

assert not current_row.empty
assert bool(current_row.iloc[0]["analysed_deep"])
assert bool(current_row.iloc[0]["analysed_profile"])
assert bool(current_row.iloc[0]["analysed_superficial"])

print(
    "Varianti:",
    current_row.iloc[0]["analysis_variants"],
)


## 11. Griglia risolta e catena di difficoltà


In [ ]:
deep_result = mode_results["deep"]

sv.draw_grid(
    deep_result["solved_grid"],
    given_mask=(deep_result["original"] != 0),
    title=(
        f"{deep_result['name']} - "
        f"{deep_result['grading']['label']}"
    ),
)
plt.show()

sv.plot_difficulty_chain(deep_result)


## 12. Passaggio più difficile


In [ ]:
chain_dataframe = sv.summary_dataframe(deep_result)

if chain_dataframe.empty:
    print("Nessun passaggio registrato.")
else:
    hardest_rows = (
        chain_dataframe
        .sort_values(
            ["difficolta", "step"],
            ascending=[False, True],
        )
        .head(10)
    )
    display(hardest_rows)

    hardest_index = int(hardest_rows.iloc[0]["step"]) - 1
    sv.draw_step(deep_result, hardest_index)


## 13. Catena completa e frequenza delle tecniche


In [ ]:
display(chain_dataframe)

if not chain_dataframe.empty:
    technique_counts = (
        chain_dataframe
        .groupby(
            ["strategia", "famiglia", "tecnica"],
            dropna=False,
        )
        .agg(
            numero_step=("step", "count"),
            difficolta_massima=("difficolta", "max"),
            conclusioni=("conclusioni", "sum"),
            risultati_distinti=("risultati_distinti", "sum"),
            prove=("prove", "sum"),
        )
        .reset_index()
        .sort_values(
            ["difficolta_massima", "numero_step"],
            ascending=[False, False],
        )
    )

    display(technique_counts)


## 14. Le quattro heatmap principali

Le viste `deep` usano l intero inventario salvato. Le viste `superficial`
usano la sola frontiera alla difficoltà minima dello stesso risultato deep.


In [ ]:
core_heatmaps = [
    ("deep", "extended"),
    ("deep", "compact"),
    ("superficial", "extended"),
    ("superficial", "compact"),
]

heatmap_dataframes = {}

for depth, view in core_heatmaps:
    print(f"depth={depth}, view={view}")

    heatmap_dataframes[(depth, view)] = (
        sv.technique_activity_dataframe(
            deep_result,
            depth=depth,
            view=view,
            metric="conclusions",
        )
    )

    sv.plot_technique_activity(
        deep_result,
        depth=depth,
        view=view,
        metric="conclusions",
        scale="log",
        annotate="auto",
    )


## 15. Riepilogo numerico delle heatmap


In [ ]:
heatmap_rows = []

for (depth, view), dataframe in heatmap_dataframes.items():
    values = dataframe.to_numpy()
    heatmap_rows.append({
        "depth": depth,
        "view": view,
        "righe": dataframe.shape[0],
        "step": dataframe.shape[1],
        "somma": int(values.sum()) if values.size else 0,
        "massimo": int(values.max()) if values.size else 0,
        "righe_attive": int((dataframe.sum(axis=1) > 0).sum()),
    })

summary_heatmaps = pd.DataFrame(heatmap_rows)
display(summary_heatmaps)

for key, dataframe in heatmap_dataframes.items():
    print("\n", key)
    display(
        dataframe.assign(totale=dataframe.sum(axis=1))
        .sort_values("totale", ascending=False)
        .head(20)
    )


## 16. Conclusioni, risultati distinti e prove


In [ ]:
if RUN_ALL_METRIC_HEATMAPS:
    for metric in ("conclusions", "outcomes", "proofs"):
        print("Metrica:", metric)
        sv.plot_technique_activity(
            deep_result,
            depth="deep",
            view="compact",
            metric=metric,
            scale="log",
            annotate="auto",
        )
else:
    print("Confronto metriche disattivato.")


## 17. Scale cromatiche


In [ ]:
if RUN_ALL_SCALE_HEATMAPS:
    for scale in ("log", "sqrt", "linear"):
        print("Scala:", scale)
        sv.plot_technique_activity(
            deep_result,
            depth="deep",
            view="compact",
            metric="conclusions",
            scale=scale,
            annotate="auto",
        )
else:
    print("Confronto scale disattivato.")


## 18. Vista tecnica granulare


In [ ]:
if RUN_TECHNIQUE_DETAIL_HEATMAP:
    sv.plot_technique_activity(
        deep_result,
        depth="deep",
        view="technique",
        metric="conclusions",
        scale="log",
        annotate=False,
        show_totals=True,
    )
else:
    print("Vista tecnica disattivata.")


## 19. Inventario realmente registrato da profile e superficial

Qui `depth="deep"` significa tutto ciò che è contenuto nella singola analisi.
Per `profile` è la fascia configurata; per `superficial` è soltanto la frontiera.


In [ ]:
for mode in ("profile", "superficial"):
    result = mode_results[mode]
    print("Analisi salvata:", mode)

    sv.plot_technique_activity(
        result,
        depth="deep",
        view="compact",
        metric="conclusions",
        scale="log",
        annotate="auto",
    )


## 20. Picchi di attività logica


In [ ]:
deep_compact = sv.technique_activity_dataframe(
    deep_result,
    depth="deep",
    view="compact",
    metric="conclusions",
)

if deep_compact.empty:
    print("Nessun dato di attività.")
else:
    print("Strategie più attive")
    display(
        deep_compact.sum(axis=1)
        .sort_values(ascending=False)
        .rename("conclusioni_totali")
        .to_frame()
    )

    print("Step con maggiore attività")
    display(
        deep_compact.sum(axis=0)
        .sort_values(ascending=False)
        .rename("conclusioni_totali")
        .to_frame()
        .head(15)
    )


## 21. Riepilogo delle tre analisi


In [ ]:
display(
    sv.analyses_summary_dataframe([
        mode_results["deep"],
        mode_results["profile"],
        mode_results["superficial"],
    ])
)


## 22. Galleria delle modalità


In [ ]:
sv.gallery(
    [
        mode_results["deep"],
        mode_results["profile"],
        mode_results["superficial"],
    ],
    solved=True,
    ncols=3,
)


## 23. Batch opzionale dell archivio

Anche il batch passa da `analyse_puzzle_cached`, quindi riutilizza le analisi
deep già presenti e salva quelle mancanti.


In [ ]:
if RUN_ARCHIVE_BATCH:
    references = sa.list_sudokus(
        number=ARCHIVE_BATCH_LIMIT,
        method="latest",
    )
    batch_results = []

    for reference in references:
        try:
            analysis = sa.analyse_puzzle_cached(
                reference["id"],
                analysis_mode="deep",
                force=False,
            )
            batch_results.append(analysis)
        except Exception as error:
            print(
                reference["name"],
                type(error).__name__,
                error,
            )

    if batch_results:
        display(
            sv.analyses_summary_dataframe(batch_results)
            .sort_values(
                "difficolta_massima",
                ascending=False,
            )
        )
else:
    print("Batch archivio disattivato.")


## 24. Controlli finali


In [ ]:
assert chain_signature(mode_results["deep"]) == chain_signature(
    mode_results["profile"]
)
assert chain_signature(mode_results["deep"]) == chain_signature(
    mode_results["superficial"]
)

assert all(
    step["n_proofs"] >= step["n_distinct_outcomes"]
    for step in deep_result["chain"]
)
assert all(
    step["n_conclusions"] >= 1
    for step in deep_result["chain"]
)

final_row = pd.DataFrame(
    sa.list_sudokus(method="all")
)
final_row = final_row[final_row["id"] == puzzle_reference]

assert not final_row.empty
assert bool(final_row.iloc[0]["analysed_deep"])
assert bool(final_row.iloc[0]["analysed_profile"])
assert bool(final_row.iloc[0]["analysed_superficial"])

print("TUTTI I CONTROLLI FINALI SONO STATI SUPERATI.")


## File prodotti dall archivio

Per ogni Sudoku la struttura attesa è simile a:

```text
sudoku_data/
    puzzles/
        <puzzle_id>.json
    analyses/
        <puzzle_id>/
            analysis.json
            analysis_profile_1.json
            analysis_superficial.json
```

Una finestra profile diversa genera una variante diversa, per esempio
`analysis_profile_0p5.json`, senza cancellare quella con finestra 1.0.
